In [176]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.model_selection import train_test_split

In [177]:
df = pd.read_csv('https://raw.githubusercontent.com/aniruddhachoudhury/Red-Wine-Quality/refs/heads/master/winequality-red.csv')

In [178]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [179]:
X = df.iloc[:, -3:-1].to_numpy()
Y = df.iloc[:, -1:].to_numpy()

XTrain, XTest, YTrain, YTest = train_test_split(X, Y, test_size=20)

In [180]:
class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        self.layer1 = nn.Linear(2,3)
        self.layer2 = nn.Linear(3, 1)
        self.relu = nn.ReLU()

        W1 = torch.tensor([[0.1, 0.05], [0.2, 0.02], [0.15, 0.15]])
        W2 = torch.tensor([[0.15, 0.07, 0.02]])
        B1 = torch.tensor([1.0, 1.0, 1.0])
        B2 = torch.tensor([1.0])

        with torch.no_grad():
            self.layer1.weight.copy_(W1)
            self.layer1.bias.copy_(B1)
            self.layer2.weight.copy_(W2)
            self.layer2.bias.copy_(B2)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        print('x -> layer2', x)
        x = self.layer2(x) # hàm kích hoạt ko để layer cuối cùng,tới layer 2 là trả về lun
        return x

In [181]:
def loss(YHat, Y):
    # return (0.5 * ( YHat - Y) ** 2).mean()
    #SE
    return ((YHat - Y)**2).mean()

In [182]:
import torch.optim as optim
model = Perceptron()
# optimizer = optim.Adam(model.parameters(), lr=0.01) # hàm tối ưu để chạy gradient về 0
optimizer = optim.SGD(model.parameters(), lr=0.01) # mặc định momentum =0, triệt tiêu hệ số phạt.


In [183]:
model.train()

# từ đây trở xuống là bước huấn luyện mô hình

for _ in range(10):
    for i, x in enumerate(XTrain):
        print(x)
        YHat = model(torch.tensor(x, dtype=torch.float32))
        print('y - yhat', torch.tensor(YTrain[i], dtype=torch.float32) - YHat)
        l = loss(YHat, torch.tensor(YTrain[i], dtype=torch.float32))
        optimizer.zero_grad()
        print(list(model.parameters()))
        l.backward() # đánh dấu chỗ backward
        optimizer.step()
        print(list(model.parameters()))
        break
    print(l)
    break

[ 0.49 10.8 ]
x -> layer2 tensor([1.5890, 1.3140, 2.6935], grad_fn=<ReluBackward0>)
y - yhat tensor([4.6158], grad_fn=<SubBackward0>)
[Parameter containing:
tensor([[0.1000, 0.0500],
        [0.2000, 0.0200],
        [0.1500, 0.1500]], requires_grad=True), Parameter containing:
tensor([1., 1., 1.], requires_grad=True), Parameter containing:
tensor([[0.1500, 0.0700, 0.0200]], requires_grad=True), Parameter containing:
tensor([1.], requires_grad=True)]
[Parameter containing:
tensor([[0.1068, 0.1996],
        [0.2032, 0.0898],
        [0.1509, 0.1699]], requires_grad=True), Parameter containing:
tensor([1.0138, 1.0065, 1.0018], requires_grad=True), Parameter containing:
tensor([[0.2967, 0.1913, 0.2687]], requires_grad=True), Parameter containing:
tensor([1.0923], requires_grad=True)]
tensor(21.3056, grad_fn=<MeanBackward0>)


## Cách 2: bỏ 1 vòng for

In [184]:
model.train()

for _ in range(10):
    YHat = model(torch.tensor(XTrain, dtype = torch.float32))
    l = loss(YHat, torch.tensor(YTrain[i], dtype = torch.float32))
    optimizer.zero_grad()
    l.backward() # đánh dấu chỗ backward
    optimizer.step()
    print(l)

x -> layer2 tensor([[3.2213, 2.0758, 2.9111],
        [3.2327, 2.1553, 2.9496],
        [3.2509, 2.1030, 2.9417],
        ...,
        [3.0189, 2.0095, 2.7484],
        [2.9462, 1.9582, 2.6793],
        [3.2669, 2.1335, 2.9644]], grad_fn=<ReluBackward0>)
tensor(7.8607, grad_fn=<MeanBackward0>)
x -> layer2 tensor([[5.1043, 3.2899, 4.6162],
        [5.0865, 3.3506, 4.6282],
        [5.1521, 3.3289, 4.6633],
        ...,
        [4.7141, 3.1025, 4.2834],
        [4.5884, 3.0171, 4.1663],
        [5.1698, 3.3604, 4.6874]], grad_fn=<ReluBackward0>)
tensor(0.3611, grad_fn=<MeanBackward0>)
x -> layer2 tensor([[4.5823, 2.9509, 4.1430],
        [4.5726, 3.0170, 4.1624],
        [4.6250, 2.9867, 4.1855],
        ...,
        [4.2442, 2.7974, 3.8574],
        [4.1332, 2.7215, 3.7536],
        [4.6422, 3.0179, 4.2092]], grad_fn=<ReluBackward0>)
tensor(0.4328, grad_fn=<MeanBackward0>)
x -> layer2 tensor([[5.1108, 3.2944, 4.6222],
        [5.0930, 3.3551, 4.6342],
        [5.1587, 3.3334, 4.6693],
 

## Test

In [185]:
model.eval() # bước này là bước test

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

with torch.no_grad(): # ko cập nhật gradient, ko xây dựng đồ thị
    YHat = model(torch.tensor(XTest, dtype = torch.float32))
    mae = mean_absolute_error(YTest, YHat)
    mse = mean_squared_error(YTest, YHat)
    r2 = r2_score(YTest, YHat)
    
    print(f'MAE: {mae}, MSE: {mse}, R2: {r2}')

x -> layer2 tensor([[4.0160, 2.6346, 3.6433],
        [4.0587, 2.6717, 3.6863],
        [4.0275, 2.6554, 3.6591],
        [4.8149, 3.1132, 4.3640],
        [4.3442, 2.8659, 3.9530],
        [4.9087, 3.1217, 4.4300],
        [5.2419, 3.2812, 4.7148],
        [4.5639, 2.9814, 4.1449],
        [4.4357, 2.8703, 4.0158],
        [4.7779, 3.0865, 4.3289],
        [4.4367, 2.9124, 4.0329],
        [4.0160, 2.6346, 3.6433],
        [4.3952, 2.8775, 3.9915],
        [4.5848, 2.9789, 4.1578],
        [4.0761, 2.6628, 3.6945],
        [3.9627, 2.6189, 3.6016],
        [4.1200, 2.7020, 3.7391],
        [5.3505, 3.3569, 4.8170],
        [4.4078, 2.9004, 4.0089],
        [4.4785, 2.9073, 4.0588]])
MAE: 0.7181889414787292, MSE: 0.8922603726387024, R2: -0.48710060119628906
